# Análisis de Calidad: Cartera y Pagos - Paigo
**Autor:** Franco Michele Robotti Alonso  
**Fecha:** 15 de mayo de 2026  
**Objetivo:** Carga, normalización y validación de datasets para el análisis de riesgo crediticio.

---
### Configuración del Entorno
En esta etapa, realizamos la ingesta de datos y el tipado correcto de las fechas para evitar errores de procesamiento.

### 1. Importación y Carga de Datos
*   **Librerías:** Se utiliza `duckdb` para consultas SQL de alto rendimiento sobre los DataFrames de `pandas`.
*   **Normalización:** Se fuerza el formato `datetime` en las columnas de fechas para asegurar la consistencia en los cálculos posteriores de mora[cite: 1].

In [7]:
import os
import duckdb
import numpy as np
import openpyxl
import pandas as pd

In [8]:
file_path = '../data/raw/datasets_riesgo_v2.xlsx'

df_cartera = pd.read_excel(file_path, sheet_name='cartera', decimal=',')
df_pagos = pd.read_excel(file_path, sheet_name='pagos', decimal=',')

df_pagos['fecha_vencimiento'] = pd.to_datetime(df_pagos['fecha_vencimiento'])
df_pagos['fecha_pago_real'] = pd.to_datetime(df_pagos['fecha_pago_real'])
df_cartera['fecha_originacion'] = pd.to_datetime(df_cartera['fecha_originacion'])

print(f"Cartera cargada: {df_cartera.shape[0]} registros.")
print(f"Pagos cargados: {df_pagos.shape[0]} registros.")

Cartera cargada: 400 registros.
Pagos cargados: 1349 registros.


## 2. Auditoría y Calidad de Datos

Antes de proceder con el análisis financiero y de riesgo, es imperativo validar la robustez estructural del dataset. Esta fase garantiza que las conclusiones obtenidas se basen en datos fiables, íntegros y consistentes.

**Alcance de la Auditoría:**

*   **Integridad Referencial:** Validación de las relaciones entre tablas para asegurar la ausencia de registros huérfanos.
*   **Consistencia Temporal:** Verificación de la cronología lógica entre fechas de originación y fechas de pago.
*   **Auditoría de Duplicados:** Confirmación de la unicidad de cada préstamo y pago en el sistema.
*   **Análisis de Integridad (Nulos):** Evaluación de datos faltantes y su impacto en la lógica de negocio.
*   **Consistencia Interna:** Validación de la coherencia entre el saldo del préstamo, su estado actual y el comportamiento de pago real.

### 2.1 Integridad Referencial
Se ha validado la relación entre `df_pagos` y `df_cartera`.

> **Resultado:** Se confirma una **integridad del 100%**. Todos los registros de pago están correctamente vinculados a un préstamo existente, garantizando la ausencia de datos huérfanos.

In [8]:
pagos_sin_prestamo = """
SELECT 
    p.id_prestamo,
    p.monto_pagado,
    p.fecha_vencimiento
FROM df_pagos p
LEFT JOIN df_cartera c ON p.id_prestamo = c.id_prestamo
WHERE c.id_prestamo IS NULL
"""

df_pagos_sin_prestamo = duckdb.query(pagos_sin_prestamo).df()
df_pagos_sin_prestamo

total_pagos = len(df_pagos)
pagos_huerfanos = len(df_pagos_sin_prestamo)
integridad = ((total_pagos - pagos_huerfanos) / total_pagos) * 100

print(f"Integridad Referencial: {integridad}%")

Integridad Referencial: 100.0%


### 2.2 Consistencia Cronológica
Verificamos que no existan pagos registrados con fechas anteriores a la originación del crédito.

> **Resultado:** Integridad cronológica confirmada. No se detectaron anomalías temporales en el dataset, lo que valida la calidad de la información para el análisis de riesgo.

In [9]:
fechas_de_pago_anteriores_a_originales = """
SELECT 
    p.id_prestamo,
    p.fecha_pago_real,
    c.fecha_originacion
FROM df_pagos p
JOIN df_cartera c ON p.id_prestamo = c.id_prestamo
WHERE p.fecha_pago_real < c.fecha_originacion
"""

df_fechas_anteriores = duckdb.query(fechas_de_pago_anteriores_a_originales).df()
df_fechas_anteriores

,id_prestamo,fecha_pago_real,fecha_originacion


### 2.3 Auditoría de Duplicados
Verificamos que no existan registros duplicados que puedan inflar las métricas de cartera o distorsionar los cálculos de riesgo.

**Proceso de verificación:**
* **Detección:** Buscamos registros donde coincida el identificador único (`id_prestamo` en cartera y `id_pago` en pagos).
* **Validación:** Confirmamos la unicidad de cada registro para garantizar la precisión de los cálculos financieros.

> **Resultado:** No se encontraron IDs duplicados en la tabla de cartera ni en la tabla de pagos, lo que indica que cada préstamo y cada pago están representados de manera única en el dataset. Esto es crucial para garantizar la precisión de cualquier análisis posterior, ya que los registros duplicados podrían distorsionar los resultados y llevar a conclusiones erróneas.

In [ ]:
IDs_duplicados_cartera = """
SELECT 
    id_prestamo,
    COUNT(*) AS conteo
FROM df_cartera
GROUP BY id_prestamo
HAVING COUNT(*) > 1
"""
df_ids_duplicados = duckdb.query(IDs_duplicados_cartera).df()
df_ids_duplicados

IDs_duplicados_pagos = """
SELECT 
    id_pago,
    COUNT(*) AS conteo
FROM df_pagos
GROUP BY id_pago
HAVING COUNT(*) > 1
"""
df_ids_duplicados = duckdb.query(IDs_duplicados_pagos).df()
df_ids_duplicados

,id_pago,conteo


### 2.4. Análisis de Integridad y Valores Nulos
Tras verificar la unicidad de los registros, procedemos a analizar la presencia de valores nulos en `fecha_pago_real`. Estos valores no representan errores de carga, sino información latente sobre el comportamiento de pago.

#### 2.4.1. Diagnóstico de Inconsistencias
Al cruzar los datos de la cartera con el histórico de pagos, detectamos préstamos que, aun presentando cuotas impagas vencidas, mantienen un estado de "Al día" en el sistema original.

> **Hallazgo Clave:** Identificamos 60 préstamos con inconsistencias críticas. Estos casos presentan cuotas impagas desde años anteriores (2022-2024) que no están siendo reflejadas en el estado de mora del sistema.

In [11]:
print("Nulos en Cartera:")
print(df_cartera.isnull().sum())

print("\nNulos en Pagos:")
print(df_pagos.isnull().sum())

Nulos en Cartera:
id_prestamo          0
id_cliente           0
producto             0
fecha_originacion    0
monto_original       0
plazo_meses          0
tna                  0
segmento_cliente     0
canal_adquisicion    0
pais                 0
estado_actual        0
dias_mora            0
saldo_capital        0
dtype: int64

Nulos en Pagos:
id_pago                0
id_prestamo            0
nro_cuota              0
fecha_vencimiento      0
monto_cuota            0
fecha_pago_real      151
monto_pagado           0
dias_atraso          151
dtype: int64


In [12]:
nulos = """
SELECT 
    p.*,
    c.*
FROM df_pagos as p
join df_cartera c ON p.id_prestamo = c.id_prestamo
WHERE p.fecha_pago_real IS NULL
ORDER BY p.fecha_vencimiento ASC
LIMIT 10;
"""
df_nulos = duckdb.query(nulos).df()
df_nulos

,id_pago,id_prestamo,nro_cuota,fecha_vencimiento,monto_cuota,fecha_pago_real,monto_pagado,dias_atraso,id_prestamo_1,id_cliente,...,fecha_originacion,monto_original,plazo_meses,tna,segmento_cliente,canal_adquisicion,pais,estado_actual,dias_mora,saldo_capital
0,51224,10366,1,2022-02-03,1383.75,NaT,0.0,NaN,10366,1160,...,2022-01-04,2700,2,0.30,Inactivo,Alianza Comercial,AR,Al día,0,2272.07
1,51006,10308,1,2022-02-16,3605.33,NaT,0.0,NaN,10308,1040,...,2022-01-17,52000,15,0.48,Nuevo,App Store,AR,Cancelado,0,0.00
2,51225,10366,2,2022-03-05,1383.75,NaT,0.0,NaN,10366,1160,...,2022-01-04,2700,2,0.30,Inactivo,Alianza Comercial,AR,Al día,0,2272.07
3,50659,10205,1,2022-03-11,7995.00,NaT,0.0,NaN,10205,1116,...,2022-02-09,7800,1,0.30,Recurrente,Referido,PY,31-60 días mora,56,4105.68
4,50621,10194,1,2022-03-14,1787.83,NaT,0.0,NaN,10194,1197,...,2022-02-12,17000,10,0.62,Nuevo,Alianza Comercial,AR,Al día,0,16212.77
5,50444,10143,1,2022-03-15,210.33,NaT,0.0,NaN,10143,1220,...,2022-02-13,2400,12,0.62,Inactivo,Paid Social,PY,Al día,0,2229.71
6,50384,10122,2,2022-03-23,4120.00,NaT,0.0,NaN,10122,1201,...,2022-01-22,20000,5,0.36,Nuevo,Alianza Comercial,UY,31-60 días mora,38,19798.78
7,50854,10261,1,2022-03-29,3075.00,NaT,0.0,NaN,10261,1111,...,2022-02-27,6000,2,0.30,Nuevo,Paid Social,PY,31-60 días mora,33,3299.24
8,51049,10320,2,2022-04-11,891.43,NaT,0.0,NaN,10320,1101,...,2022-02-10,6000,7,0.48,Nuevo,Alianza Comercial,AR,Cancelado,0,0.00
9,51117,10336,1,2022-04-17,5098.50,NaT,0.0,NaN,10336,1060,...,2022-03-18,19800,4,0.36,Nuevo,Paid Social,AR,Al día,0,19162.37


In [13]:
inconsistencia_mora = """
SELECT 
    c.id_prestamo,
    c.estado_actual,
    c.dias_mora,
    COUNT(p.nro_cuota) AS cuotas_vencidas_no_pagas,
    MIN(p.fecha_vencimiento) AS vencimiento_mas_antiguo
FROM df_cartera c
JOIN df_pagos p ON c.id_prestamo = p.id_prestamo
WHERE p.fecha_pago_real IS NULL 
  AND p.fecha_vencimiento < '2026-05-14'
  AND c.estado_actual = 'Al día'
GROUP BY ALL
"""

df_alarmas = duckdb.query(inconsistencia_mora).df()
print(f"Total de préstamos con inconsistencia: {len(df_alarmas)}")
df_alarmas

Total de préstamos con inconsistencia: 60


,id_prestamo,estado_actual,dias_mora,cuotas_vencidas_no_pagas,vencimiento_mas_antiguo
0,10004,Al día,0,2,2022-07-07
1,10074,Al día,0,1,2023-03-10
2,10087,Al día,0,1,2024-05-21
3,10150,Al día,0,1,2023-09-28
4,10171,Al día,0,1,2023-09-07
5,10153,Al día,0,1,2024-05-01
6,10194,Al día,0,3,2022-03-14
7,10304,Al día,0,1,2023-11-09
8,10337,Al día,0,2,2024-01-18
9,10346,Al día,0,1,2023-05-07


#### 2.4.2. Clasificación y Cuantificación de Nulos
**Análisis de Hallazgos:**
* **Mora Real Reconocida (73 registros):** Representan el 48.3% de los nulos. Son registros con consistencia entre tablas, donde la cuota está vencida y el préstamo correctamente identificado en la cartera. El capital asociado es de $343,422.93.
* **Inconsistencias de Mora (78 registros):** Representan el 51.7% de los nulos. Son cuotas vencidas (desde febrero de 2022) no pagadas, pero cuyo préstamo figura erróneamente como "Al día". Esta anomalía afecta a 60 préstamos únicos y compromete un capital de $263,668.14 que no está siendo provisionado correctamente.
* **Ausencia de Cuotas Futuras:** Se confirmó que 0 registros corresponden a cuotas vigentes o futuras, lo que indica que el dataset es una fotografía de obligaciones exigibles que permanecen impagas.

In [14]:
analisis_restante_nulos = """
SELECT 
    CASE 
        WHEN p.fecha_vencimiento >= '2026-05-14' THEN 'Cuotas Futuras (Vigentes)'
        WHEN p.fecha_vencimiento < '2026-05-14' AND c.estado_actual != 'Al día' THEN 'Mora Correcta'
        WHEN p.fecha_vencimiento < '2026-05-14' AND c.estado_actual = 'Al día' THEN 'Inconsistencia'
        ELSE 'Otros'
    END AS tipo_nulo,
    COUNT(*) AS cantidad,
    SUM(p.monto_cuota) AS capital_asociado,
    MIN(p.fecha_vencimiento) AS vencimiento_min,
    MAX(p.fecha_vencimiento) AS vencimiento_max,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) || '%' AS porcentaje
FROM df_pagos p
LEFT JOIN df_cartera c ON p.id_prestamo = c.id_prestamo
WHERE p.fecha_pago_real IS NULL
GROUP BY 1
ORDER BY 1
"""

df_detalle_151 = duckdb.query(analisis_restante_nulos).df()
df_detalle_151

,tipo_nulo,cantidad,capital_asociado,vencimiento_min,vencimiento_max,porcentaje
0,Inconsistencia,78,263668.14,2022-02-03,2024-10-31,51.66%
1,Mora Correcta,73,343422.93,2022-02-16,2024-12-03,48.34%


### 2.5. Validación de Consistencia Interna
Verificamos la coherencia entre las variables de saldo y el estado del préstamo en la cartera.

#### 2.5.1. Hallazgos de Consistencia
Realizamos dos pruebas de integridad sobre el dataset:
* **Consistencia de Mora:** Verificamos si existen préstamos con `dias_mora > 0` marcados como "Al día".
* **Consistencia de Saldo:** Buscamos préstamos con `saldo_capital = 0` que no se encuentren en estado "Cancelado".

> **Resultado:** Ambas pruebas arrojaron resultados satisfactorios (0 casos de error), confirmando que la cartera es consistente consigo misma. No obstante, esto refuerza que el problema detectado anteriormente (la inconsistencia con el histórico de pagos) no es un error de formato, sino una discrepancia entre procesos de negocio ya que la tabla cartera es consistente consigo misma, pero la tabla cartera no es consistente con la realidad de los pagosporque ignoró las cuotas vencidas de esos 60 préstamos anteriores.

In [15]:
inconsistencia_campos_cartera = """
SELECT 
    id_prestamo,
    pais,
    producto,
    estado_actual,
    dias_mora,
    saldo_capital
FROM df_cartera
WHERE dias_mora > 0 
  AND estado_actual = 'Al día'
"""

df_error_campos = duckdb.query(inconsistencia_campos_cartera).df()
print(f"Casos con dias_mora > 0 y estado 'Al día': {len(df_error_campos)}")
df_error_campos

Casos con dias_mora > 0 y estado 'Al día': 0


,id_prestamo,pais,producto,estado_actual,dias_mora,saldo_capital


In [16]:
inconsistencia_saldo_final = """
SELECT 
  c.*
FROM df_cartera as c
WHERE ROUND(saldo_capital, 2) = 0 
  AND estado_actual NOT IN ('Cancelado')
"""

df_errores_saldo = duckdb.query(inconsistencia_saldo_final).df()
print(f"Hallazgo: {len(df_errores_saldo)} préstamos que no deben nada pero siguen activos.")
df_errores_saldo

Hallazgo: 0 préstamos que no deben nada pero siguen activos.


,id_prestamo,id_cliente,producto,fecha_originacion,monto_original,plazo_meses,tna,segmento_cliente,canal_adquisicion,pais,estado_actual,dias_mora,saldo_capital


#### 2.5.2. Definición de la "Fuente de Verdad"
Para proceder con la corrección de los datos, establecemos los siguientes criterios técnicos:
* **Fuente de Verdad:** La tabla de pagos es la fuente de verdad absoluta para determinar la mora real.
* **Criterio de Mora Efectiva:** Se define como mora cualquier préstamo con al menos una cuota de vencimiento anterior al corte (2026-05-14) y `fecha_pago_real` nula.
* **Hipótesis de Error:** Los 60 casos inconsistentes se asumen como una falla en la actualización automática de los procesos del sistema.

In [ ]:
mora_por_prestamo = duckdb.query("""
    SELECT 
        id_prestamo,
        COUNT(*) AS cuotas_vencidas,
        MAX(current_date - fecha_vencimiento) AS dias_mora_calculados,
        SUM(monto_cuota) AS capital_en_mora
    FROM df_pagos
    WHERE fecha_pago_real IS NULL 
      AND fecha_vencimiento < '2026-05-14'
    GROUP BY id_prestamo
""").df()

df_cartera_final = df_cartera.merge(mora_por_prestamo, on='id_prestamo', how='left')

df_cartera_final['estado_analista'] = np.where(
    df_cartera_final['cuotas_vencidas'] > 0, 
    'Mora Reclasificada', 
    df_cartera_final['estado_actual']
)

df_cartera_final['dias_mora_calculados'] = df_cartera_final['dias_mora_calculados'].fillna(0)